In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1onRUGy8gu0P3yeps5GLGkxKRfFKFp2kU", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_setup.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# System Prompts & The Right Altitude

*Part 1 of the Vizuara series on Prompt Design Principles*
*Estimated time: 45 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/prompt-design-principles/practice/1/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_matter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Install Deps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_install_deps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Every interaction with a Large Language Model begins with a prompt. But here is the surprising truth: most LLM failures are not the model's fault — they are **prompt design failures**. A vague prompt gives vague results. A rigid prompt breaks on edge cases. And the difference between a toy demo and a production system often comes down to how well you design your prompts.

In this notebook, we will build practical intuition for what makes a prompt "good" and master the single most important prompt design concept: **the Right Altitude principle** for system prompts.

By the end, you will:
- Understand the three dimensions of prompt quality
- Write system prompts at the "right altitude"
- Compare responses from vague, balanced, and rigid prompts empirically
- Build a system prompt evaluation framework

Let us start with a simple experiment.

In [ ]:
# 🔧 Setup — install dependencies
!pip install -q anthropic matplotlib seaborn pandas numpy

import os
import json
import textwrap
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Set your API key (use Colab secrets or paste directly)
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("✅ Setup complete!")

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Imagine you get into a taxi and say: "Take me somewhere nice for dinner."

The driver might take you to a fast-food place, a sushi bar, or a steakhouse three towns over. Now imagine you say instead: "Take me to the Italian restaurant on 5th Street — the one with outdoor seating. Please avoid the highway."

Same driver, same car, wildly different outcomes. The second instruction is not smarter — it is **more precise**. It gives the driver enough context to make the right decisions while leaving room for judgment on the exact route.

This is the difference between a bad prompt and a good prompt. The LLM is the driver. Your prompt is the directions.

### Three Dimensions of Prompt Quality

A good prompt optimizes across three dimensions:

1. **Clarity** — The model should never have to guess what you want
2. **Specificity** — Constrain the output space to exactly what you need
3. **Structure** — Organize information so the model can parse it efficiently

### 🤔 Think About This

Consider these two prompts for a code review task:
- Prompt A: "Review this code."
- Prompt B: "You are a senior Python developer. Review this code for correctness, performance, and security. For each issue, state the problem, explain why it matters, and suggest a fix."

Which one will produce a more useful review? Why? What specific aspects of Prompt B make it better?

In [ ]:
#@title 🎧 Listen: Math Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_math_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

While prompt design is more art than pure math, we can formalize what "good" means.

Given a prompt $p$ and a desired output distribution $D^*$, the quality of a prompt can be thought of as:

$$Q(p) = -D_{\text{KL}}\big(D^* \| D_{\text{model}}(p)\big)$$

where $D_{\text{KL}}$ is the KL divergence between the desired output distribution and what the model actually produces given prompt $p$.

**What does this mean computationally?** A perfect prompt ($Q = 0$) means the model's output distribution exactly matches what you want. A vague prompt creates a wide, spread-out distribution (high entropy) — the model could produce many different outputs. A precise prompt creates a narrow, peaked distribution around your desired output.

Let us plug in some simple numbers. Suppose you want the model to classify sentiment as "positive" or "negative". With a vague prompt, the model assigns:
- $P(\text{positive}) = 0.55$, $P(\text{negative}) = 0.45$

The entropy is: $H = -(0.55 \log_2 0.55 + 0.45 \log_2 0.45) = 0.993$ bits — nearly maximum uncertainty.

With a well-designed prompt (clear task, examples, format), the model assigns:
- $P(\text{positive}) = 0.95$, $P(\text{negative}) = 0.05$

The entropy is: $H = -(0.95 \log_2 0.95 + 0.05 \log_2 0.05) = 0.286$ bits — much more certain. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Entropy Viz Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_entropy_viz_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Entropy Viz After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_entropy_viz_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 📊 Visualization: Entropy of prompt quality
import numpy as np
import matplotlib.pyplot as plt

def entropy(p):
    """Binary entropy function."""
    if p == 0 or p == 1:
        return 0
    return -(p * np.log2(p) + (1-p) * np.log2(1-p))

probs = np.linspace(0.01, 0.99, 100)
entropies = [entropy(p) for p in probs]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(probs, entropies, 'b-', linewidth=2)

# Mark vague prompt
ax.axvline(x=0.55, color='red', linestyle='--', alpha=0.7)
ax.annotate('Vague prompt\nP(correct)=0.55\nH=0.99 bits',
            xy=(0.55, entropy(0.55)), xytext=(0.3, 0.6),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')

# Mark good prompt
ax.axvline(x=0.95, color='green', linestyle='--', alpha=0.7)
ax.annotate('Good prompt\nP(correct)=0.95\nH=0.29 bits',
            xy=(0.95, entropy(0.95)), xytext=(0.75, 0.6),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=11, color='green')

ax.set_xlabel('P(correct answer)', fontsize=12)
ax.set_ylabel('Entropy (bits)', fontsize=12)
ax.set_title('A Good Prompt Reduces Output Uncertainty', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Section Transition Build
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_section_transition_build.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 The Prompt Quality Evaluator

Let us build a framework that sends the same task to the LLM with different prompt styles and compares the results.

In [ ]:
#@title 🎧 Listen: Query Llm Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_query_llm_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from anthropic import Anthropic

client = Anthropic()

def query_llm(system_prompt, user_message, model="claude-sonnet-4-20250514"):
    """Send a message to Claude with a given system prompt."""
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

# Test with a simple query
result = query_llm(
    system_prompt="You are a helpful assistant.",
    user_message="What is batch normalization in one sentence?"
)
print("Response:", result)

In [ ]:
#@title 🎧 Listen: Three Altitude Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_three_altitude_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Three Altitude Levels

Now let us define system prompts at three altitude levels for a **code review** task:

In [ ]:
#@title 🎧 Listen: Define Prompts Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_define_prompts_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Altitude Level 1: Too Vague (30,000 ft) ---
VAGUE_PROMPT = "You are a helpful assistant. Answer questions accurately."

# --- Altitude Level 2: Right Altitude (10,000 ft) ---
RIGHT_ALTITUDE_PROMPT = """You are a senior Python developer acting as a code reviewer.
Review code for: correctness, performance, and security.
For each issue found:
- State the problem clearly
- Explain why it matters
- Suggest a specific fix with code

Be direct and constructive. Skip praise — focus on what needs fixing.
If the code is clean, say so in one sentence."""

# --- Altitude Level 3: Too Rigid (Ground Level) ---
RIGID_PROMPT = """If the user submits Python code with a for loop, check if it can
be replaced with a list comprehension. If yes, output exactly:
"SUGGESTION: Replace for loop on line {N} with list comprehension."
If the code has more than 50 lines, split your review into sections
labeled SECTION_1, SECTION_2, etc.
If the user asks a follow-up question, respond only about the most
recent code block.
Never mention security unless the code uses eval() or exec().
Always end with "END_REVIEW"."""

print("✅ Three prompt levels defined!")
print(f"  Vague:          {len(VAGUE_PROMPT):>4} chars")
print(f"  Right Altitude: {len(RIGHT_ALTITUDE_PROMPT):>4} chars")
print(f"  Rigid:          {len(RIGID_PROMPT):>4} chars")

In [ ]:
#@title 🎧 Listen: Run Experiment Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_run_experiment_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 Running the Experiment

In [ ]:
#@title 🎧 Listen: Run Prompts Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_run_prompts_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Code snippet to review — contains multiple issues
CODE_TO_REVIEW = '''
def get_user_data(user_id):
    import sqlite3
    conn = sqlite3.connect("users.db")
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = conn.execute(query).fetchone()
    data = []
    for i in range(len(result)):
        data.append(result[i])
    conn.close()
    return data
'''

user_message = f"Please review this Python code:\n```python\n{CODE_TO_REVIEW}\n```"

# Run all three prompts
print("=" * 70)
print("VAGUE PROMPT RESPONSE:")
print("=" * 70)
vague_response = query_llm(VAGUE_PROMPT, user_message)
print(vague_response[:800])

print("\n" + "=" * 70)
print("RIGHT ALTITUDE PROMPT RESPONSE:")
print("=" * 70)
right_response = query_llm(RIGHT_ALTITUDE_PROMPT, user_message)
print(right_response[:800])

print("\n" + "=" * 70)
print("RIGID PROMPT RESPONSE:")
print("=" * 70)
rigid_response = query_llm(RIGID_PROMPT, user_message)
print(rigid_response[:800])

In [ ]:
#@title 🎧 Listen: Compare Metrics Viz Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_compare_metrics_viz_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Compare Metrics Viz After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_compare_metrics_viz_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 📊 Visualization: Compare response quality metrics
def score_review(response):
    """Simple heuristic scoring of a code review response."""
    scores = {}
    response_lower = response.lower()

    # Did it find the SQL injection?
    scores["SQL Injection Found"] = 1 if "sql injection" in response_lower or "parameterized" in response_lower else 0

    # Did it find the performance issue?
    scores["Performance Issue Found"] = 1 if "list comprehension" in response_lower or "enumerate" in response_lower or "range(len" in response_lower else 0

    # Did it suggest specific fixes?
    scores["Specific Fixes"] = 1 if "```" in response or "def " in response else 0

    # Is it well-structured?
    scores["Structured Response"] = 1 if response.count("\n") > 5 else 0

    return scores

results = {
    "Vague (30K ft)": score_review(vague_response),
    "Right Altitude (10K ft)": score_review(right_response),
    "Rigid (Ground)": score_review(rigid_response),
}

df = pd.DataFrame(results).T
fig, ax = plt.subplots(figsize=(10, 5))
df.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black')
ax.set_ylabel("Score (0 or 1)", fontsize=12)
ax.set_title("Code Review Quality by Prompt Altitude Level", fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 1.3)
plt.tight_layout()
plt.show()

## 5. 🔧 Your Turn

### TODO 1: Write a Right-Altitude System Prompt

Write a system prompt for a **data analysis assistant** that:
- Has a clear role definition
- Specifies the output format (use markdown tables and bullet points)
- Sets appropriate constraints without being rigid
- Includes behavioral guidelines

In [ ]:
#@title 🎧 Listen: Todo1 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo1_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_data_analyst_prompt():
    """
    Write a system prompt for a data analysis assistant at the
    'right altitude' — specific enough to guide, flexible enough
    to handle varied requests.

    Requirements:
    - Define the role clearly
    - Specify output format preferences
    - Set behavioral guidelines
    - Keep it under 200 words
    """
    # ============ TODO ============
    # Write your system prompt here.
    # Think about: What role? What format? What constraints?
    # What should the assistant do vs NOT do?
    # ==============================

    system_prompt = """???"""  # YOUR PROMPT HERE

    return system_prompt

# Test your prompt
prompt = build_data_analyst_prompt()
test_query = "Analyze this data: Sales were $50K in Jan, $65K in Feb, $48K in Mar, $72K in Apr."
response = query_llm(prompt, test_query)
print("Your assistant's response:")
print(response)

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Evaluate Prompt Robustness

Test whether your prompt handles **edge cases** gracefully — this is the key
test of whether you hit the right altitude.

In [ ]:
#@title 🎧 Listen: Todo2 After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo2_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
edge_cases = [
    "What's the weather like today?",              # Off-topic
    "Can you write a poem about data?",            # Creative request
    "SELECT * FROM sales WHERE month='January'",   # SQL instead of description
    "Summarize this in French: Revenue was $100K.", # Language switch
]

# ============ TODO ============
# For each edge case, run your prompt and evaluate:
# - Does it stay in character?
# - Does it handle the unexpected input gracefully?
# - Does it break or produce garbage?
#
# Print each edge case and the response
# ==============================

for i, case in enumerate(edge_cases):
    print(f"\n--- Edge Case {i+1}: {case[:50]}... ---")
    response = "???"  # YOUR CODE: call query_llm with your prompt
    print(response[:200])

In [ ]:
#@title 🎧 Listen: Section Transition Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_section_transition_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

In [ ]:
#@title 🎧 Listen: Full Eval Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_full_eval_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Full prompt evaluation pipeline
def evaluate_prompt_suite(system_prompt, test_cases, label="Custom"):
    """
    Evaluate a system prompt across multiple test cases.

    Returns a DataFrame with scores for each test case.
    """
    results = []
    for query, expected_traits in test_cases:
        response = query_llm(system_prompt, query)
        score = sum(1 for trait in expected_traits if trait.lower() in response.lower())
        results.append({
            "query": query[:50],
            "score": score / len(expected_traits),
            "response_length": len(response),
        })
    return pd.DataFrame(results)

# Define test cases: (query, list_of_expected_traits_in_response)
code_review_tests = [
    ("Review: `data = eval(input())`",
     ["security", "eval", "dangerous", "alternative"]),
    ("Review: `for i in range(len(lst)): print(lst[i])`",
     ["idiomatic", "enumerate", "pythonic"]),
    ("Review: `import *`",
     ["namespace", "explicit", "import"]),
    ("What's your favorite color?",
     ["code", "review", "not"]),  # Should redirect to its role
]

# Evaluate all three altitude levels
for name, prompt in [("Vague", VAGUE_PROMPT),
                      ("Right Altitude", RIGHT_ALTITUDE_PROMPT),
                      ("Rigid", RIGID_PROMPT)]:
    df = evaluate_prompt_suite(prompt, code_review_tests, name)
    avg_score = df["score"].mean()
    print(f"{name:>15}: avg score = {avg_score:.2f}")

In [ ]:
#@title 🎧 Listen: Section Transition Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_section_transition_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

Since this notebook focuses on prompt design rather than model training,
our "training" is the iterative refinement of prompts. Let us visualize
how prompt iterations improve quality.

In [ ]:
#@title 🎧 Listen: Iteration Viz Before
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_iteration_viz_before.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Iteration Viz After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_iteration_viz_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Simulate prompt iteration improvement
iterations = ["v1\n(Vague)", "v2\n(Added role)", "v3\n(Added format)",
              "v4\n(Added constraints)", "v5\n(Right Altitude)"]
quality_scores = [0.35, 0.55, 0.72, 0.85, 0.93]
coverage_scores = [0.80, 0.75, 0.70, 0.68, 0.88]

fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#2196F3'
color2 = '#FF9800'

ax1.bar([i - 0.15 for i in range(5)], quality_scores, 0.3,
        label='Quality Score', color=color1, alpha=0.8, edgecolor='black')
ax1.bar([i + 0.15 for i in range(5)], coverage_scores, 0.3,
        label='Edge Case Coverage', color=color2, alpha=0.8, edgecolor='black')

ax1.set_xlabel('Prompt Iteration', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Prompt Quality Improves Through Iterative Refinement', fontsize=14)
ax1.set_xticks(range(5))
ax1.set_xticklabels(iterations, fontsize=10)
ax1.legend(loc='upper left')
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: Quality steadily improves, but edge case coverage")
print("can DIP when prompts become too specific (v3, v4) before")
print("recovering at the right altitude (v5).")

In [ ]:
#@title 🎧 Listen: Section Transition Final
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_section_transition_final.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. 🎯 Final Output

In [ ]:
#@title 🎧 Listen: Final Template Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_final_template_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build and display the ultimate system prompt template
TEMPLATE = """
╔══════════════════════════════════════════════════════════════╗
║                 SYSTEM PROMPT TEMPLATE                       ║
║                 (Right Altitude Pattern)                      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. ROLE DEFINITION                                          ║
║     "You are a [specific role] with expertise in [domain]."  ║
║                                                              ║
║  2. TASK SCOPE                                               ║
║     "[Action verb] [object] for [criteria]."                 ║
║                                                              ║
║  3. OUTPUT FORMAT                                            ║
║     "For each [item], provide:                               ║
║      - [Field 1]                                             ║
║      - [Field 2]                                             ║
║      - [Field 3]"                                            ║
║                                                              ║
║  4. BEHAVIORAL GUIDELINES                                    ║
║     "Be [tone]. [Do/Don't] [specific behavior]."            ║
║                                                              ║
║  5. EDGE CASE HANDLING                                       ║
║     "If [condition], [action]."                              ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(TEMPLATE)

# Example using the template
example_prompt = """You are a senior machine learning engineer acting as a technical interviewer.

Evaluate the candidate's response to ML system design questions.

For each response, provide:
- Technical accuracy (1-5)
- Depth of understanding (1-5)
- Communication clarity (1-5)
- A specific follow-up question to probe deeper

Be encouraging but honest. Focus on understanding, not memorization.
If the candidate's answer is off-topic, gently redirect to the question."""

print("\n📝 Example prompt using the template:")
print("-" * 50)
print(example_prompt)

print("\n🎉 Congratulations! You've mastered the Right Altitude principle!")
print("You can now write system prompts that are specific yet flexible.")

In [ ]:
#@title 🎧 Listen: Reflection Next Steps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_reflection_next_steps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### 🤔 Reflection Questions
1. Why does adding too many rigid rules (ground level) often **reduce** output quality rather than improve it?
2. How would you adapt the Right Altitude principle for a customer support chatbot vs. a code review tool?
3. What signals tell you that your prompt is at the wrong altitude?

### 🏆 Optional Challenges
1. **A/B Test**: Take a real prompt you use regularly and write a "right altitude" version. Compare outputs on 5 different inputs.
2. **Prompt Library**: Create a collection of 5 right-altitude system prompts for different tasks (summarization, translation, analysis, creative writing, debugging).
3. **Altitude Detector**: Write a function that heuristically scores whether a system prompt is too vague, too rigid, or right altitude based on word count, specificity markers, and if-else patterns.